# Answer verification: letting a model check a model

> In the previous lecture we let the model **sample** the same problem many times, then used self-consistency to majority-vote across the answers. Voting counts how often an answer appears; it does not inspect the reasoning of each answer. When wrong answers cluster, majority vote selects the wrong answer.
>
> This lecture adds a check after sampling: an independent **verifier** scores each candidate, and we keep the highest-scoring one. We will see how a verifier learns to score from data, what is gained and lost by checking only the final answer versus checking every reasoning step, and whether a verifier can still be trained without human labels.

We start with a minimal example. On the same problem, the model gives three candidate solutions:

- Solution A: every reasoning step is correct, final answer 17.
- Solution B: one intermediate step is wrong, final answer 18.
- Solution C: one intermediate step is also wrong, final answer 18.

The correct answer is 17. **Majority vote** counts only final answers: 17 appears once, 18 appears twice, so 18 wins. The correct answer is 17, so the vote is wrong. Two incorrect solutions happen to share the same answer and outvote the correct one.

A different approach uses an independent **verifier** to score each solution from 0 to 1: solution A has clean reasoning and scores 0.9; solution B errs midway and scores 0.2; solution C likewise scores 0.2. We pick the highest score — solution A, answer 17, which is correct.

The difference is the selector. Voting counts how often an answer appears, so a cluster of errors pulls it off course. The verifier scores the quality of each solution on its own, so a clean reasoning trace can surface even when its answer is not the majority. This lecture implements that verifier: score the candidates and pick the highest.

The need for the check is the same phenomenon as last lecture. Asked the same hard problem several times, the same model is sometimes right and sometimes wrong. Last lecture we used that randomness to sample repeatedly and then vote, but voting does not inspect reasoning, so a cluster of wrong answers wins — the example above is that failure in miniature.

Below we build the verifier on a toy task that can be graded automatically, use it for best-of-N selection, and compare checking only the final answer with checking every reasoning step. The first step is to quantify the loss that occurs when a correct answer is generated and then discarded by a bad choice.

## 1. The generation-verification gap

Last lecture we sampled repeatedly and voted across the answers. This section turns from generation to selection: after many candidates have been sampled, how much is lost in the choice. Seeing that loss tells us whether a verification stage is worth adding.

Autoregressive models need verification because they emit one token at a time and cannot go back to fix a mistake. Once an intermediate step is wrong, later steps are built on that error and the whole solution is contaminated. When wrong answers cluster, majority vote fails as well.

The Weaver paper quantifies this with two quantities. `Pass@K` is the fraction of problems whose candidate pool contains at least one correct solution, measuring whether a correct answer was generated. SuccessRate is the probability that the selector actually picks a correct one, measuring whether the choice is right. Their difference is the generation-verification gap.

Measuring both quantities needs an environment that can grade automatically. We encode uniqueness of the answer and programmatic grading into a toy task: each problem is two sub-expressions plus a combining operator, and the final answer is the combination of the two sub-results. The solution text has a fixed format, so it can be parsed and recomputed line by line.

## 2. How a verifier learns to score from answers

This section explains where a verifier's scoring ability comes from: it is learned from data. We walk through the procedure of Cobbe et al., then use the toy task to see what noise enters the automatic labels.

The procedure comes from the paper that released GSM8K. Training has three stages. First the generator is fine-tuned on the training problems so it can solve them. Then 100 candidate solutions are sampled from the generator for each problem, and each is labeled positive or negative by whether the final answer equals the gold answer. Finally an independent verifier is trained on that automatically labeled set. No human reads any solution.

The verifier and the generator are two independent networks. The verifier can emit a probability of correctness at the end of a solution, scoring the whole trace; it can also predict that probability after every token, using the last token's score at test time. The second version is finer-grained. For now we treat the verifier as giving one score at the end of the solution; a finer version appears later.

Automatic labeling has a clear noise source. A solution whose intermediate steps are wrong but whose final answer happens to be correct (accidentally correct) is labeled positive; a solution whose intermediate steps are all correct but whose final answer is written wrong is labeled negative. When the label and the reasoning quality disagree, that is noise. We measure that noise on the toy task.

## 3. Outcome verifiers and process verifiers

This section compares giving a supervision signal to the whole solution versus giving one to each step, and shows what checking only the final answer misses.

The outcome verifier of the previous section looks only at whether the final answer is correct: one label per solution. That design has a hidden cost. A negative label says only that something went wrong, not which step. An error on step one and an error on step two receive the same label. On hard problems most candidates contain errors, so these negative labels carry little information. This is the credit-assignment problem: one score for the whole solution cannot assign blame to a specific step.

A natural idea is to make the supervision as fine as each step. Each step of the reasoning chain gets a probability of being correct, so which step dragged the solution down is written into the labels. That verifier is a process verifier (PRM, process-supervised reward model). The previous kind, which gives one label to the whole solution, is an outcome verifier (ORM, outcome-supervised reward model). Both names come from Lightman et al., 2023.

The table below lists four combinations of intermediate steps and final answer, and the labels each form of supervision assigns:

| Intermediate steps | Final answer | Outcome label | Process label |
|---|---|---|---|
| all correct | correct | positive | every step positive |
| some wrong | correct (accidentally correct) | positive | contains a negative step |
| all correct | wrong (wrong ending) | negative | every step positive |
| some wrong | wrong | negative | contains a negative step |

The middle two rows matter. Outcome supervision labels "wrong steps, right answer" as positive and "right steps, wrong answer" as negative, so the label comes apart from reasoning quality. Process supervision writes the truth of each step directly, so the location of the error is clear.

## 4. Checking at the granularity of each step

This section measures how much is gained by moving the check from the outcome to each step, on real data and on our toy task.

Lightman et al. (2023) moved verification from judging the outcome to judging the process. A process verifier outputs, for each step of the reasoning chain, the probability that the step is correct. The score of the whole solution is reduced by a product: the probability that every step is correct. If a label is neutral (cannot tell right from wrong), it is treated as positive.

On MATH, best-of-1860, PRM reaches 78.2%, the outcome verifier ORM 72.4%, and majority vote 69.6%. The gap widens as the number of candidates N grows. Training a PRM used PRM800K: 800k human step-level labels, annotated only up to the first error, balancing cost and information.

We reproduce that main line on the toy task: train two verifiers, one with outcome labels (one per solution) and one with process labels (one per step), and compare the scores they learn inside best-of-N. Here we restore the consistency features of Section 2 to raw numbers, so both verifiers face the same amount of information.

## 5. What to do without step-by-step human labels

Step-level labels are expensive, so this section looks at whether they can be produced without humans. We examine how Math-Shepherd does automatic labeling, and what new problems automatic labels introduce.

Process supervision needs a human label on every step. PRM800K has 800k labels, which is costly. Math-Shepherd (Wang et al., 2023) replaces humans with automatic labels. The idea is that the quality of a step is determined by whether a correct answer can still be reached from that step, not by whether the step itself was computed correctly. A completer continues from the current intermediate step for N subsequent paths, and the step is labeled by how many of those paths reach the gold answer.

Automatic labeling has two estimators, each with a trade-off. The next paragraphs work through the numbers, then we reproduce the pattern with a scripted completer.

## 6. Combining the verifier with sampling

This section covers how the verifier pairs with the sampling of the previous lecture, and what to do when a single verifier is not reliable enough.

Repeated sampling from last lecture aims to put at least one correct solution in the candidate pool. The verifier's job is to pick that correct one from the pool. Together they are best-of-N: sample N candidate solutions, score each with the verifier, and keep the highest-scoring one. Two details in the experiments are worth watching: whether an extra step after taking the top score still helps, and how much better this is than majority vote.

When a single weak verifier is not reliable enough, several verifiers can be combined with weights. Weaver (Saad-Falcon et al., 2025) needs no human labels: it recovers each verifier's accuracy from how much the verifiers agree with each other, then aggregates with naive Bayes. Details wait for the experiment below. The conclusion to keep is that an ensemble of weak verifiers can approach a stronger one.

Verification signals can be lined up by source: programmatic ground truth (an executor or the final answer), outcome supervision (one label for the whole solution), process supervision (a label per step), and multi-verifier ensembles (no labels). Cost rises, and so does robustness. The next lecture looks at a harder verification signal: run the reasoning inside a tool and take the execution result as ground. Lecture 6 on reinforcement learning will use this lecture's verifier as a reward signal; the quality of the reward sets the ceiling of training.

## Summary

- [ ] Generation is easy, verification is hard: autoregressive generation has no correction mechanism, so one wrong intermediate step contaminates the whole solution
- [ ] The generation-verification gap is Pass@K − SuccessRate; majority vote fails when wrong answers cluster
- [ ] Cobbe's verifier training: sample from the generator → label automatically by the final answer → train an independent verifier → best-of-N at test time
- [ ] Automatic outcome labels have two kinds of noise: accidentally correct (wrong steps, right answer) and wrong ending (right steps, wrong answer)
- [ ] Outcome supervision (ORM) uses one label for the whole solution and has a credit-assignment problem; process supervision (PRM) gives each step a label
- [ ] A PRM's solution score is reduced by a product or a minimum, and in best-of-N it clearly beats majority vote and the outcome verifier
- [ ] Math-Shepherd uses a completer to auto-label steps; HE introduces false positives when N is large, SE is more stable
- [ ] Several weak verifiers can be weighted with method-of-moments, without labels, approaching the oracle's selection ability

## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.

Each of the three fill-in exercises includes a reference solution and can be run as-is. Cover the answers first, write your own, then compare.

**Exercise 1: two reductions for a PRM solution score**

Given an array of per-step correctness probabilities for one solution, complete the product reduction and the min reduction.

Hint: product reduction multiplies every step's probability, so longer solutions score lower; min reduction looks only at the worst step, and a wrong step is exposed immediately.


**Exercise 2: HE and SE in Math-Shepherd**

Given whether each of N continuation paths from an intermediate step is finally correct, complete hard estimation and soft estimation.

Hint: HE is existence (1 if any path is True); SE is frequency (the fraction of True). Smaller N makes SE noisier.

**Exercise 3: verifier selection vs majority vote**

Given a set of candidates, each with (verifier score, final answer, whether correct), complete the two selectors: pick the highest score, and majority-vote the answers.

Hint: majority vote counts how often an answer appears and ignores verifier scores; when wrong answers cluster but have low scores, the verifier wins.


## References

- Cobbe et al., [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168), 2021 — founding paper on verifiers and GSM8K; establishes outcome verification and the best-of-N framework
- Lightman et al., [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050), 2023 — systematic comparison of outcome and process supervision; releases PRM800K
- Wang et al., [Math-Shepherd: Verify and Reinforce LLMs Step-by-step without Human Annotations](https://arxiv.org/abs/2312.08935), 2023 — auto-labels steps with a completer (HE/SE) and does step-by-step PPO
- Saad-Falcon et al., [Shrinking the Generation-Verification Gap with Weak Verifiers](https://arxiv.org/abs/2506.18203), NeurIPS 2025 — unlabeled method-of-moments and weak-verifier ensembles (Weaver)
- Uesato et al., [Solving Math Word Problems with Process- and Outcome-based Feedback](https://arxiv.org/abs/2211.14275), 2022 — first comparison of outcome and process
- Li et al., [Making Language Models Better Reasoners with Step-Aware Verifier](https://arxiv.org/abs/2210.01241), 2022 — NLI/rule auto step-labeling; a baseline for Math-Shepherd
- Wang et al., [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171), 2022 — majority-vote baseline (introduced last lecture)
- Hendrycks et al., [Measuring Mathematical Problem Solving with the MATH Dataset](https://arxiv.org/abs/2103.03874), 2021 — the MATH dataset
- Shao et al., [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300), 2024 — GRPO; continuation of step-level reward in reinforcement learning

We first look at what a problem in this toy task looks like. Each problem is two sub-expressions plus a combining operator.
Prompt: 3 + 4 and 2 * 5, combining operator +. The gold answer is computed in two steps: first 3 + 4 = 7 and 2 * 5 = 10, then 7 + 10 = 17.
A candidate solution is three lines of fixed-format text. The first two lines are the results of the two sub-expressions; the third is the final answer:

```text
3 + 4 = 7
2 * 5 = 10
Answer = 17
```

Whether this solution is good can be judged by recomputing each line: recompute whether 3 + 4 equals 7, whether 2 * 5 equals 10, and whether the final answer equals the gold 17. Each solution therefore yields two boolean arrays: whether each step is computed correctly (step_ok) and whether the final answer is written correctly (final_ok).
A unique, programmatically gradable answer is the key property of the toy task: it lets us label any solution automatically, without a human. The code below randomly generates such problems and solutions; the gold answer of each problem is uniquely determined by the program.

In [ ]:
import numpy as np
import re

OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b, "*": lambda a, b: a * b}


def apply(op, a, b):
    """Apply one arithmetic operation and return an integer result."""
    return OPS[op](a, b)


def make_problem(rng):
    """Sample a two-part arithmetic problem; return the prompt, combining operator, and gold answer."""
    a1 = int(rng.integers(1, 10))
    op1 = str(rng.choice(["+", "-", "*"]))
    b1 = int(rng.integers(1, 10))
    if op1 == "-" and a1 < b1:
        a1, b1 = b1, a1
    a2 = int(rng.integers(1, 10))
    op2 = str(rng.choice(["+", "-", "*"]))
    b2 = int(rng.integers(1, 10))
    if op2 == "-" and a2 < b2:
        a2, b2 = b2, a2
    cop = str(rng.choice(["+", "-"]))
    golden = apply(cop, apply(op1, a1, b1), apply(op2, a2, b2))
    return {"parts": [(a1, op1, b1), (a2, op2, b2)],
            "combine": cop, "golden": golden}


def slip(rng):
    """Sample a small random arithmetic slip (plus or minus 1 to 3)."""
    return int(rng.integers(1, 4)) * int(rng.choice([-1, 1]))


def sample_solution(problem, step_acc, rng):
    """Generate a candidate with a scripted solver of tunable accuracy.

    Each intermediate step is correct with probability step_acc. Correctness of
    the final answer is derived from the intermediate steps, then a little end
    noise is added. Returns (text, per-step correctness, final correctness,
    final answer value).
    """
    (a1, op1, b1), (a2, op2, b2) = problem["parts"]
    cop = problem["combine"]
    golden = problem["golden"]
    l_true = apply(op1, a1, b1)
    r_true = apply(op2, a2, b2)
    l_p = l_true if rng.random() < step_acc else l_true + slip(rng)
    r_p = r_true if rng.random() < step_acc else r_true + slip(rng)
    derived = (l_p == l_true) and (r_p == r_true)
    eps, lucky = 0.08, 0.08
    final_ok = (derived and rng.random() > eps) or \
               (not derived and rng.random() < lucky)
    if final_ok:
        a_p = golden
    else:
        a_p = golden - 1 if rng.random() < 0.8 else golden + slip(rng)
    text = "\n".join([
        f"{a1} {op1} {b1} = {l_p}",
        f"{a2} {op2} {b2} = {r_p}",
        f"Answer = {a_p}",
    ])
    step_ok = [l_p == l_true, r_p == r_true]
    return text, step_ok, final_ok, a_p


rng = np.random.default_rng(42)
for _ in range(2):
    p = make_problem(rng)
    text, step_ok, final_ok, _ = sample_solution(p, 0.5, rng)
    print("Gold answer:", p["golden"])
    print(text)
    print("Step correctness:", step_ok, "| final correctness:", final_ok)
    print("---")

In [ ]:
def judge_solution(text, problem):
    """Split the solution text into steps, recompute true values line by line, and compare with the gold answer.

    Returns (step_ok, final_ok): whether each step is correct and whether the
    final answer is correct.
    """
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    (a1, op1, b1), (a2, op2, b2) = problem["parts"]
    golden = problem["golden"]
    expect = [apply(op1, a1, b1), apply(op2, a2, b2), golden]
    step_ok = []
    for line in lines:
        m = re.match(r"^(-?\d+)\s*([+\-*])\s*(-?\d+)\s*=\s*(-?\d+)$", line)
        if m is None:
            continue
        i = len(step_ok)
        c = int(m.group(4))
        step_ok.append(c == expect[i])
    final_ok = None
    for line in lines:
        m = re.match(r"^Answer = (-?\d+)$", line)
        if m:
            final_ok = int(m.group(1)) == golden
    return step_ok, final_ok


rng = np.random.default_rng(7)
for _ in range(2):
    p = make_problem(rng)
    text, _, _, _ = sample_solution(p, 0.5, rng)
    step_ok, final_ok = judge_solution(text, p)
    print("Gold answer:", p["golden"])
    print(text)
    print("Program judge: steps", step_ok, "| final", final_ok)
    print("---")

The difference between Pass@K and SuccessRate is clear from one problem and three solutions.
Suppose K = 3. For the same problem (gold answer 17) we draw three solutions:

```text
Solution A: 3 + 4 = 7, 2 * 5 = 10, Answer = 17   (final correct)
Solution B: 3 + 4 = 8, 2 * 5 = 10, Answer = 18   (final wrong)
Solution C: 3 + 4 = 8, 2 * 5 = 10, Answer = 18   (final wrong)
```

The candidate pool contains a correct solution: A is in the pool, so Pass@3 = 1; a correct answer really was generated.
Majority-voting the three solutions by final answer: 18 appears twice, 17 once, so the vote selects 18, while the correct answer is 17. Selection fails on this problem, SuccessRate is 0.
Pass@3 − SuccessRate = 1 − 0 = 1. That is the generation-verification gap. The gap is not a loss of generation ability — the correct answer is already in the pool — but a loss in selection: the selector only counts how often an answer appears, and is overwritten by a cluster of wrong answers. The verifier fills that slot. It does not count frequency; it evaluates the quality of each solution on its own.

In [ ]:
def extract_answer(text):
    """Extract the final-answer number from the solution text."""
    for line in text.strip().splitlines():
        m = re.match(r"^Answer = (-?\d+)$", line.strip())
        if m:
            return int(m.group(1))
    return None


def pass_at_k(demos, K, step_acc, seed=1):
    """For each problem, sample K solutions and measure the fraction whose pool has at least one correct final answer."""
    rng = np.random.default_rng(seed)
    ok = 0
    for p in demos:
        hit = any(sample_solution(p, step_acc, rng)[2] for _ in range(K))
        ok += int(hit)
    return ok / len(demos)


def majority_success(demos, K, step_acc, seed=1):
    """For each problem, sample K solutions, majority-vote the most frequent answer, and measure the hit rate."""
    from collections import Counter
    rng = np.random.default_rng(seed)
    ok = 0
    for p in demos:
        votes = [sample_solution(p, step_acc, rng)[3] for _ in range(K)]
        top = Counter(votes).most_common(1)[0][0]
        ok += int(top == p["golden"])
    return ok / len(demos)


rng = np.random.default_rng(42)
demos = [make_problem(rng) for _ in range(40)]
K = 25
pk = pass_at_k(demos, K, step_acc=0.5)
mj = majority_success(demos, K, step_acc=0.5)
print("Pass@K (correct solution in the pool): {:.2f}".format(pk))
print("Majority-vote SuccessRate:             {:.2f}".format(mj))
print("Generation-verification gap:           {:.2f}".format(pk - mj))

Cobbe's verifier training splits into three steps, each depending only on automatic judgment, with no human in the loop.
The pipeline starts with the generator: it is fine-tuned on the training data so it can solve the training problems. Then come three steps.
Step one, sampling. For each training problem, the trained generator draws 100 candidate solutions. The generator is stochastic, so the solutions drawn for the same problem can differ.
Step two, automatic labeling. Each solution gets one label: 1 if the final answer equals the gold answer, else 0. This step compares a single integer and is fully automatic. On the three solutions above, A is labeled 1, B and C are labeled 0.
Step three, training the verifier. Pairs of (solution features, label) become ordinary supervised data. An independent network is trained to predict the probability that the solution is finally correct. At test time this verifier scores new candidates.
Sampling, labeling, and training are all automatic. That is the definition of outcome supervision: the supervision signal is only the final result of the whole solution.
Automatic labels are not noise-free. A solution with wrong intermediate steps whose final answer happens to equal the gold (accidentally correct) is mislabeled positive; a solution with all-correct intermediate steps whose last line is wrong is mislabeled negative. Below we count both kinds of noise in the data.

In [ ]:
rng = np.random.default_rng(42)
n_probs = 200
cpp = 25
step_acc = 0.5
problems = [make_problem(rng) for _ in range(n_probs)]

pool = []
for p in problems:
    for _ in range(cpp):
        text, step_ok, final_ok, a = sample_solution(p, step_acc, rng)
        pool.append(dict(prob=p, text=text, step_ok=step_ok,
                         final_ok=final_ok, answer=a))

n_clean = sum(1 for d in pool if all(d["step_ok"]) and d["final_ok"])
n_lucky = sum(1 for d in pool if not all(d["step_ok"]) and d["final_ok"])
n_misend = sum(1 for d in pool if all(d["step_ok"]) and not d["final_ok"])
print("Candidate total:", len(pool))
print("All steps correct and final answer correct:", n_clean, "({:.2f})".format(n_clean / len(pool)))
print("Some step wrong but final answer correct (accidentally correct):", n_lucky,
      "({:.2f})".format(n_lucky / len(pool)))
print("All steps correct but final answer wrong (wrong ending):", n_misend,
      "({:.2f})".format(n_misend / len(pool)))

# Show one accidentally correct solution
lucky_demo = next(d for d in pool if not all(d["step_ok"]) and d["final_ok"])
print("\nAccidentally correct example:")
print(lucky_demo["text"])
print("Gold answer:", lucky_demo["prob"]["golden"])

In [ ]:
def step_consistency(text):
    """Parse literal consistency of each step in the text (recompute whether A op B == C)."""
    cons = []
    for line in text.strip().splitlines():
        line = line.strip()
        m = re.match(r"^(-?\d+)\s*([+\-*])\s*(-?\d+)\s*=\s*(-?\d+)$", line)
        if m:
            a, op, b, c = (int(m.group(1)), m.group(2),
                           int(m.group(3)), int(m.group(4)))
            cons.append(1.0 if apply(op, a, b) == c else 0.0)
    return cons


def orm_features(text):
    """Input features for the outcome verifier: per-step consistency, the answer value, and a constant.

    The per-step consistency flags come from parse-and-recompute, so the
    verifier head also checks the arithmetic of each step.
    """
    cons = step_consistency(text)
    ans = 0.0
    for line in text.strip().splitlines():
        m = re.match(r"^Answer = (-?\d+)$", line.strip())
        if m:
            ans = int(m.group(1)) / 100.0
    return np.array(cons[:2] + [ans, 1.0], dtype=np.float32)


n_tr = 140
tr_mask = np.zeros(len(pool), bool)
for i in range(n_tr):
    tr_mask[i * cpp:(i + 1) * cpp] = True
te_mask = ~tr_mask

X = np.stack([orm_features(d["text"]) for d in pool])
Yf = np.array([1.0 if d["final_ok"] else 0.0 for d in pool], dtype=np.float32)
Xtr, Ytr, Xte, Yte = X[tr_mask], Yf[tr_mask], X[te_mask], Yf[te_mask]
print("Training set:", Xtr.shape, "test set:", Xte.shape)


import torch
import torch.nn as nn

torch.manual_seed(0)


def make_mlp(d_in, hidden=32):
    """Build a three-layer fully connected network that outputs one logit."""
    return nn.Sequential(
        nn.Linear(d_in, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden // 2), nn.ReLU(),
        nn.Linear(hidden // 2, 1),
    )


def train_mlp(model, Xt_, Yt_, iters=2000, batch=256, lr=0.005):
    """Train an MLP on (features, labels) and return the per-iteration loss list."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    Xt = torch.tensor(Xt_)
    Yt = torch.tensor(Yt_).unsqueeze(1)
    n = len(Xt)
    losses = []
    for _ in range(iters):
        idx = torch.randperm(n)[:batch]
        opt.zero_grad()
        loss = loss_fn(model(Xt[idx]), Yt[idx])
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


def predict_prob(model, X_):
    """Score a feature matrix with the model and return sigmoid probabilities."""
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X_))).numpy().ravel()


orm = make_mlp(4, hidden=32)
losses = train_mlp(orm, Xtr, Ytr, iters=2000)
p_orm = predict_prob(orm, Xte)
acc = ((p_orm > 0.5) == Yte).mean()
print("Outcome verifier test accuracy: {:.3f}".format(acc))
print("Loss first to last: {:.3f} -> {:.3f}".format(losses[0], losses[-1]))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("BCE loss")
plt.title("Result verifier training loss")
plt.tight_layout()
plt.show()

Once the verifier is trained, at test time it is combined with sampling into best-of-N: draw N candidate solutions for the same problem, let the verifier assign each a probability of being correct, and take the answer of the highest-scoring one.
We pick the highest score rather than majority vote because the two judge different things. Majority vote counts how often an answer appears and does not inspect the reasoning quality of each solution. When wrong answers cluster (many solutions make the same error in the same place), the vote selects the wrong answer. The verifier scores each solution on its own, so a low-scoring solution is not chosen even if its answer is frequent. Recall the three solutions above: majority vote selects 18; an ideal verifier would give A a high score and B and C low scores, and select A.
Cobbe also observed an enhancement: take the top-k solutions by verifier score, then majority-vote the final answers of those k. That is usually better than taking the single highest score. The top-scoring one may only be slightly ahead, while the correct answer often appears more than once in the top-k. Both Lightman and Math-Shepherd report that combining a verifier with self-consistency beats either alone.
Below we run best-of-N with the same trained outcome verifier and compare it with majority vote. Candidates come from the test problems; the verifier was not trained on those problems.


In [ ]:
te_start = n_tr * cpp
te_answers = np.array([d["answer"] for d in pool])
te_golden = np.array([d["prob"]["golden"] for d in pool])
te_final_ok = np.array([1.0 if d["final_ok"] else 0.0 for d in pool])
n_te = (len(pool) - te_start) // cpp
print("Number of test problems:", n_te)


def best_of_n_success(N, score_fn):
    """For each test problem, take the first N candidates, pick the highest score, and measure final-answer hit rate.

    score_fn(base, N) returns the score array of the first N candidates of
    test problem base.
    """
    ok = 0
    for pi in range(n_te):
        base = pi * cpp
        idx = base + int(np.argmax(score_fn(base, N)))
        ok += int(te_answers[te_start + idx] == te_golden[te_start + idx])
    return ok / n_te


def majority_success(N):
    """Majority vote: take the first N candidates and pick the most frequent answer."""
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        votes = te_answers[base:base + N]
        counts = {}
        for v in votes:
            counts[v] = counts.get(v, 0) + 1
        top = max(counts, key=counts.get)
        ok += int(top == te_golden[base])
    return ok / n_te


for N in [10, 25]:
    mj = majority_success(N)
    orm_ok = best_of_n_success(N, lambda b, N: p_orm[b:b + N])
    print("N={}: majority vote {:.2f} | outcome verifier best-of-N {:.2f}".format(
        N, mj, orm_ok))

The difference between an outcome verifier (ORM) and a process verifier (PRM) can be shown on the same problem and the same solution.
The problem is still 3 + 4 and 2 * 5 combined by addition, gold answer 17. The model produces:

```text
3 + 4 = 8    (first step wrong, should be 7)
2 * 5 = 10   (second step correct)
Answer = 18
```

Outcome supervision looks only at the final answer: it does not equal the gold, so the whole solution gets one label 0. The outcome verifier's judgment of this solution sits on that single score bit, a 0 or 1, which says only that the outcome is wrong, not which step.
Process supervision pushes the label onto each step: the first step is wrong and labeled 0, the second is correct and labeled 1, so this solution gets two step-level labels [0, 1], which point at the error. At training time the outcome verifier has one label per solution; the process verifier has one label per step. That is the difference in supervision granularity.
At scoring time both verifiers reduce scores to one number for the whole solution. The process verifier emits a correctness probability per step, for example [0.9, 0.8, 0.2, 0.9], with step three wrong. Product reduction multiplies every step: 0.9 × 0.8 × 0.2 × 0.9 = 0.1296, the probability that every step is correct; any low step probability pushes the whole score down. Min reduction takes the minimum across steps: min(0.9, 0.8, 0.2, 0.9) = 0.2, so the worst step determines the solution score and the most suspicious step is exposed at once.
Both reductions depend on the same fact: each step score carries information about where the error is. That is exactly what outcome supervision lacks. One label for the whole solution cannot tell "first step wrong, second right" from "first right, second wrong". That missing information is the credit-assignment problem. Below we first compute the two reductions in code, then compare the two kinds of label on a real accidentally correct solution.

In [ ]:
def score_product(step_probs):
    """Product reduction: multiply the correctness probabilities of all steps as the solution score."""
    return float(np.prod(step_probs))


def score_min(step_probs):
    """Min reduction: take the minimum of the per-step correctness probabilities."""
    return float(np.min(step_probs))


step_probs = np.array([0.9, 0.8, 0.2, 0.9])
print("Four step probabilities:", step_probs)
print("Product reduction:", round(score_product(step_probs), 4))
print("Min reduction:", round(score_min(step_probs), 4))

all_ok = np.array([0.9, 0.8, 0.9, 0.9])
print("\nWhen all steps are fine, product reduction:", round(score_product(all_ok), 4),
      "| min reduction:", round(score_min(all_ok), 4))

In [ ]:
# Pick a solution with a wrong intermediate step and a correct final answer; compare the two labels
lucky = next(d for d in pool[te_start:] if not all(d["step_ok"])
             and d["final_ok"])
print("Problem:", lucky["prob"]["parts"], "| combine:",
      lucky["prob"]["combine"], "| gold answer:", lucky["prob"]["golden"])
print(lucky["text"])
step_ok, final_ok = judge_solution(lucky["text"], lucky["prob"])
print("Step ground truth:", step_ok, "| final ground truth:", final_ok)

true_step_probs = np.array(step_ok, dtype=float)
print("Outcome-supervision label: +1 (final answer correct)")
print("Process-supervision label:", [1.0 if s else 0.0 for s in step_ok])
print("Perfect process verifier, product reduction:", round(score_product(true_step_probs), 3))
print("Perfect outcome verifier (final correctness known): 1.0")

Outcome supervision has only a whole-solution label. The experiment below tests whether it can still learn the location of the error from that label.
The outcome verifier trained earlier used per-step consistency flags as features — we hand-computed whether each step recomputes as equal and fed that in, which is equivalent to stuffing process information into the features in advance. Now those flags are removed. The verifier sees only raw numbers: operands, operator, and result of each step, plus the whole-solution answer, and we train a network of the same structure.
The experiment will show that with only a whole-solution label, the verifier struggles to learn where the error is. The reason is the information in the label itself: a negative label says only that something in this solution is wrong, not whether it is step one or step two. Same structure, same number of training steps, and the error location is not learned, so accuracy falls near the always-predict-wrong baseline.
That is the credit-assignment problem — a whole-solution label dilutes the location of the error. Process supervision cuts the label down to each step and writes the location of the error into the label, which sidesteps the problem.


In [ ]:
OP_CODE = {"+": 0, "-": 1, "*": 2}


def step_features(text, k):
    """Numeric features of step k: operands and result (normalized) + operator one-hot + step index."""
    line = [ln.strip() for ln in text.strip().splitlines() if ln.strip()][k]
    m = re.match(r"^(-?\d+)\s*([+\-*])\s*(-?\d+)\s*=\s*(-?\d+)$", line)
    a, op, b, c = int(m.group(1)), m.group(2), int(m.group(3)), int(m.group(4))
    onehot = [0.0, 0.0, 0.0]
    onehot[OP_CODE[op]] = 1.0
    return [a / 50.0, b / 50.0, c / 50.0] + onehot + [k / 3.0]


def raw_solution_features(text):
    """Raw numeric features for the outcome verifier: two-step features + answer + constant, no consistency flags."""
    feats = []
    for k in range(2):
        feats += step_features(text, k)[:6]
    ans = 0.0
    for line in text.strip().splitlines():
        m = re.match(r"^Answer = (-?\d+)$", line.strip())
        if m:
            ans = int(m.group(1)) / 100.0
    return np.array(feats + [ans, 1.0], dtype=np.float32)


Xr = np.stack([raw_solution_features(d["text"]) for d in pool])
Xr_tr, Xr_te = Xr[tr_mask], Xr[te_mask]
print("Raw feature dimension:", Xr_tr.shape[1])

orm_raw = make_mlp(14, hidden=64)
_ = train_mlp(orm_raw, Xr_tr, Ytr, iters=2500)
p_raw = predict_prob(orm_raw, Xr_te)
acc_raw = ((p_raw > 0.5) == Yte).mean()
print("Outcome verifier (raw features) test accuracy: {:.3f}".format(acc_raw))
print("Baseline (always predict wrong): {:.3f}".format(1 - Yte.mean()))
print("Key observation: with only a whole-solution label, the verifier does not learn which step is wrong.")

In [ ]:
def raw_step_dataset():
    """Expand each step of each solution into an independent (features, label) sample."""
    Xs_, ys_ = [], []
    for d in pool:
        for k in range(2):
            Xs_.append(step_features(d["text"], k))
            ys_.append(1.0 if d["step_ok"][k] else 0.0)
    return np.array(Xs_, np.float32), np.array(ys_, np.float32)


Xs_all, ys_all = raw_step_dataset()
mask_rep = np.repeat(tr_mask, 2)
Xs_tr, ys_tr = Xs_all[mask_rep], ys_all[mask_rep]
Xs_te, ys_te = Xs_all[~mask_rep], ys_all[~mask_rep]
print("Step-level training samples:", Xs_tr.shape[0], "| step-level test samples:", Xs_te.shape[0])

prm = make_mlp(7, hidden=64)
_ = train_mlp(prm, Xs_tr, ys_tr, iters=2500)
p_step = predict_prob(prm, Xs_te)
acc_step = ((p_step > 0.5) == ys_te).mean()
print("Process verifier step-level accuracy: {:.3f}".format(acc_step))
print("Step-level positive rate (baseline): {:.3f}".format(ys_te.mean()))
print("Key observation: with one clean label per step, the process verifier learns each step's correctness.")

In [ ]:
p_step_test = p_step.reshape(-1, 2)


def oracle_success(N):
    """Perfect process verifier: select by the product of per-step ground truth."""
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        true_prod = [np.prod(pool[base + j]["step_ok"]) for j in range(N)]
        idx = base + int(np.argmax(true_prod))
        ok += int(te_answers[idx] == te_golden[idx])
    return ok / n_te


Ns = [1, 3, 5, 10, 15, 25]
print("N    maj    ORM(out)  PRM(proc)  perfect PRM")
for N in Ns:
    mj = majority_success(N)
    orm_ok = best_of_n_success(N, lambda b, N: p_raw[b:b + N])
    prm_ok = best_of_n_success(N, lambda b, N: p_step_test[b:b + N].prod(1))
    orc = oracle_success(N)
    print("{:<5} {:<6.2f} {:<9.2f} {:<9.2f} {:.2f}".format(
        N, mj, orm_ok, prm_ok, orc))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(Ns, [majority_success(N) for N in Ns], "o-", label="Majority vote")
plt.plot(Ns, [best_of_n_success(N, lambda b, N: p_raw[b:b + N]) for N in Ns],
         "s-", label="ORM (outcome)")
plt.plot(Ns, [best_of_n_success(N, lambda b, N: p_step_test[b:b + N].prod(1))
              for N in Ns], "^-", label="PRM (process)")
plt.plot(Ns, [oracle_success(N) for N in Ns], "d--", label="Perfect verifier")
plt.xlabel("Number of candidates N")
plt.ylabel("Success rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Pick a test solution with a wrong intermediate step and a correct final answer; look at its process scores
te_lucky = [d for d in pool[te_start:] if not all(d["step_ok"])
            and d["final_ok"]]
example = te_lucky[0]
k_ex = pool.index(example) - te_start
probs_ex = p_step_test[k_ex]
print("Solution text:\n" + example["text"])
print("Per-step PRM scores:", np.round(probs_ex, 3))
print("Product reduction:", round(score_product(probs_ex), 3))
print("Min reduction:", round(score_min(probs_ex), 3))


rows = p_step_test[:12]
plt.figure(figsize=(6, 3.5))
plt.imshow(rows, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar()
plt.xlabel("Step")
plt.ylabel("Solution index")
plt.title("Per-step correctness probability")
plt.tight_layout()
plt.show()

Math-Shepherd addresses the cost of obtaining step-level labels: the labels are useful, but labeling every step by hand is expensive, so the annotation pipeline is automated.
The angle of labeling also changes: we do not ask whether this step itself was computed correctly, but whether a correct answer can still be reached by continuing from this step. That angle comes from MCTS (Monte Carlo tree search): a completer continues from the current intermediate step for N subsequent full solutions, and we look at how many finally reach the gold answer.
A concrete number. Suppose after some step the completer continues N = 8 complete solutions, of which 3 have a final answer equal to gold and 5 do not. Hard estimation (HE) looks only at existence: if any continuation reaches the correct answer the step is labeled 1, so this step is labeled 1. Soft estimation (SE) looks at frequency: 3 / 8 = 0.375, so this step is labeled 0.375.
Both estimators are the same formula family. Write the final answer of continuation j as $a_j$ and the gold answer as $a^*$:

```text
HE: y_i = 1[there exists j with a_j = a*]
SE: y_i = (1/N) * Σ_j 1[a_j = a*]
```

The larger N, the closer SE is to this step's true correctness rate, while HE is more prone to false positives. As soon as one continuation lucks into the right answer, the step is labeled 1 even if the step itself is wrong. The authors found that the distribution of SE is closer to human labels, and HE degrades from false positives as N grows. Below we reproduce that pattern with a scripted completer.

In [ ]:
def roll_out(prefix_ok, comp_acc, rng):
    """Continue one full path from the current state; return whether the final answer equals gold."""
    p = comp_acc if prefix_ok else 0.05
    return bool(rng.random() < p)


def auto_label(prefix_ok, N, comp_acc, rng):
    """Label an intermediate step: continue N paths, return (HE, SE)."""
    hits = np.array([roll_out(prefix_ok, comp_acc, rng) for _ in range(N)])
    return int(hits.any()), float(hits.mean())


rng = np.random.default_rng(7)
n_steps = 2000
true_ok = np.array([bool(rng.random() < 0.5) for _ in range(n_steps)])
comp_acc = 0.8

agree_he_list, agree_se_list = [], []
Ns_roll = [1, 4, 8, 16]
for N in Ns_roll:
    he_arr, se_arr = [], []
    for ok in true_ok:
        he, se = auto_label(ok, N, comp_acc, rng)
        he_arr.append(he)
        se_arr.append(se)
    he_arr = np.array(he_arr)
    se_arr = np.array(se_arr)
    a_he = (he_arr == true_ok).mean()
    a_se = ((se_arr > 0.5) == true_ok).mean()
    agree_he_list.append(a_he)
    agree_se_list.append(a_se)
    print("N={}: HE agreement {:.3f} | SE (threshold 0.5) agreement {:.3f} | HE positive rate {:.3f}"
          .format(N, a_he, a_se, he_arr.mean()))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.plot(Ns_roll, agree_he_list, "o-", label="HE agreement")
plt.plot(Ns_roll, agree_se_list, "s-", label="SE agreement")
plt.xlabel("Number of rollouts N")
plt.ylabel("Agreement with true labels")
plt.legend()
plt.tight_layout()
plt.show()

The value of step-level labels is not only selection at test time; they can also serve as reinforcement-learning rewards.
Reinforcement learning is a way of training by adjusting behavior from feedback. The feedback signal is called a reward. Classical outcome supervision gives one reward at the end of the whole solution. Step-level labels allow a reward at the end of each step: a positive score if the step is correct, a negative score if it is wrong. Receiving a signal at every step is easier to learn from than receiving one signal only at the end.
Math-Shepherd used automatically labeled step-level labels for step-by-step PPO, which outperformed ORM-PPO that rewards only the final outcome. Those two training algorithms are not developed in this lecture. GRPO in lecture 6 continues the idea of training a reasoner with step-level rewards.
Below we first use SE scores for best-of-N selection, then a small example of step-level reward: the reward of the whole solution is the sum of the per-step rewards.

In [ ]:
def se_score_solution(text, prob, N, comp_acc, rng):
    """Use SE labels as step-level scores: continue N paths per step, return the product of the SEs."""
    step_ok_true, _ = judge_solution(text, prob)
    scores = []
    for k in range(2):
        prefix_ok = all(step_ok_true[:k + 1])
        _, se = auto_label(prefix_ok, N, comp_acc, rng)
        scores.append(se)
    return float(np.prod(scores))


def best_of_n_se(N_sel, N_roll):
    """Best-of-N selection using the product of SE scores."""
    rng2 = np.random.default_rng(3)
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        scores = []
        for j in range(N_sel):
            d = pool[base + j]
            scores.append(se_score_solution(d["text"], d["prob"],
                                            N_roll, comp_acc, rng2))
        idx = base + int(np.argmax(scores))
        ok += int(te_answers[idx] == te_golden[idx])
    return ok / n_te


for N in [10, 25]:
    print("N={}: majority vote {:.2f} | SE auto-label best-of-N {:.2f}".format(
        N, majority_success(N), best_of_n_se(N, 8)))

# The idea of step-by-step RL: a reward at the end of each step, not only at the end of the solution
step_rewards = np.array([0.9, 0.4, 0.85])
print("\nPer-step reward:", step_rewards)
print("Sum of rewards for the whole solution:", round(step_rewards.sum(), 2))

In [ ]:
def pass_at_k_curve(K):
    """Whether the first K candidates of each test problem contain at least one finally correct solution."""
    ok = 0
    for pi in range(n_te):
        base = te_start + pi * cpp
        ok += int(te_final_ok[base:base + K].any())
    return ok / n_te


K_vals = [2, 4, 8, 16, 25, 40]
print("K     Pass@K   maj    PRM   gap(maj) gap(PRM)")
for K in K_vals:
    pk = pass_at_k_curve(K)
    mj = majority_success(K)
    prm_ok = best_of_n_success(K, lambda b, N: p_step_test[b:b + N].prod(1))
    print("{:<5} {:<8.2f} {:<6.2f} {:<6.2f} {:<9.2f} {:.2f}".format(
        K, pk, mj, prm_ok, pk - mj, pk - prm_ok))

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(K_vals, [pass_at_k_curve(K) for K in K_vals], "o-",
         label="Pass@K (oracle in pool)")
plt.plot(K_vals, [majority_success(K) for K in K_vals], "s-",
         label="Majority vote")
plt.plot(K_vals, [best_of_n_success(K, lambda b, N: p_step_test[b:b + N].prod(1))
                  for K in K_vals], "^-", label="PRM best-of-N")
plt.xlabel("K (candidates)")
plt.ylabel("Success rate")
plt.legend()
plt.tight_layout()
plt.show()

### Estimating each verifier's accuracy without labels

When a single verifier is unreliable, several weak verifiers are ensembled. The difficulty is that, without human labels, we do not know how accurate each verifier is. Weaver's core observation is that **how often verifiers agree with each other contains their accuracies**. Once that observation is in place, the formulas are easier to read.

We first build intuition with a concrete setting. Suppose there are 100 candidate solutions, 40 correct and 60 wrong. A verifier votes "correct" or "wrong" on each. Two numbers matter: among the 40 correct ones, the fraction it votes "correct" (true positive rate, TPR; higher means it catches the correct ones); among the 60 wrong ones, the fraction it votes "wrong" (true negative rate, TNR; higher means it does not let wrong ones through). TPR and TNR together are this verifier's accuracy. The catch is that the base rate "40 are correct" requires human labels, and we assume there are none.

The way around this is that how often two verifiers vote "correct" at the same time depends on how accurate each is. Two accurate verifiers vote "correct" together on correct solutions and "wrong" together on wrong ones, so joint "correct" votes are frequent; if either is inaccurate, that agreement is scattered. So pairwise co-vote frequencies can be inverted into each verifier's accuracy, without knowing which solutions are truly correct.

That intuition becomes equations. Write the probability that a solution is truly correct as $p = P(Y=1)$ (this base rate is also unknown), and verifier k's vote as $S_k \in \{0,1\}$. Assume the verifiers do not affect each other given correctness or incorrectness (conditional independence — probabilities of independent events multiply, a fact from introductory probability). Two quantities we can count directly can then be written in the unknowns:

```text
P(S_k = 1)          = p·TPR_k + (1-p)·(1-TNR_k)
P(S_i = 1, S_j = 1) = p·TPR_i·TPR_j + (1-p)·(1-TNR_i)·(1-TNR_j)
```

The left-hand sides are frequencies countable from the data (the fraction of times verifier k votes "correct", and the fraction of times i and j vote "correct" together). The right-hand sides are a model in the unknowns p, TPR, and TNR. Connecting a countable quantity to a desired quantity with an equation is how the unknowns become solvable — using observable quantities to recover unobservable parameters is method of moments.

Whether the system can be solved depends on having enough equations. K verifiers give K unary equations (each verifier's "correct" frequency) and K(K-1)/2 pairwise equations (pairwise co-"correct" frequencies). The unknowns are one p plus each verifier's TPR and TNR, 1+2K in total. For K=4: 4 + 6 = 10 equations and 1 + 8 = 9 unknowns, more equations than unknowns, so it is solvable; with fewer than 4 verifiers there are not enough equations. Extra equations each carry random noise, so we do not seek an exact solution, but one that minimizes the error of all equations at once — least squares (as in a linear algebra course; here it is enough to know we look for the solution closest to all equations).

With each verifier's accuracy in hand, selection becomes probabilistic inference. Given a solution, the verifiers cast a vote vector s. Under conditional independence we multiply the per-verifier conditional probabilities and compute the posterior that it is truly correct:

```text
P(Y=1 | s) ∝ P(Y=1) · Π_k P(S_k = s_k | Y=1)
```

where $P(S_k=1|Y=1) = TPR_k$ and $P(S_k=0|Y=1) = 1 - TPR_k$. The highest-scoring solution is the one jointly favored by the verifiers. The code below simulates 5 weak verifiers of different accuracies, estimates TPR/TNR from unlabeled agreement only, and selects by the posterior.

In [ ]:
from scipy.optimize import least_squares

K_v = 5
true_tpr = np.array([0.62, 0.71, 0.80, 0.55, 0.88])
true_tnr = np.array([0.58, 0.66, 0.75, 0.52, 0.84])

# Simulate votes of 5 weak verifiers on the test solutions
rng = np.random.default_rng(11)
Y_te = te_final_ok[te_start:]
n_te_sol = len(Y_te)
S = np.zeros((n_te_sol, K_v))
for k in range(K_v):
    p = np.where(Y_te == 1, true_tpr[k], 1 - true_tnr[k])
    S[:, k] = (rng.random(n_te_sol) < p).astype(float)

obs1 = S.mean(0)
obs2 = (S[:, :, None] * S[:, None, :]).mean(0)
pairs = [(i, j) for i in range(K_v) for j in range(i + 1, K_v)]


def residuals(theta):
    """Moment residuals: unary moments + pairwise moments."""
    pY, tpr, tnr = theta[0], theta[1:1 + K_v], theta[1 + K_v:]
    r = [pY * tpr[k] + (1 - pY) * (1 - tnr[k]) - obs1[k]
         for k in range(K_v)]
    for i, j in pairs:
        r.append(pY * tpr[i] * tpr[j] +
                 (1 - pY) * (1 - tnr[i]) * (1 - tnr[j]) - obs2[i, j])
    return np.array(r)


x0 = np.concatenate([[0.3], np.full(2 * K_v, 0.7)])
res = least_squares(residuals, x0, bounds=(0.001, 0.999), max_nfev=20000)
pY_hat, tpr_hat, tnr_hat = res.x[0], res.x[1:1 + K_v], res.x[1 + K_v:]
print("P(Y) estimate: {:.3f} (true {:.3f})".format(pY_hat, Y_te.mean()))
print("TPR estimate:", np.round(tpr_hat, 2), "(true", true_tpr, ")")
print("TNR estimate:", np.round(tnr_hat, 2), "(true", true_tnr, ")")


def posterior_score(s, pY, tpr, tnr):
    """Naive-Bayes posterior P(Y=1|s)."""
    p1 = pY * np.prod(np.where(s == 1, tpr, 1 - tpr))
    p0 = (1 - pY) * np.prod(np.where(s == 1, 1 - tnr, tnr))
    return p1 / (p1 + p0)


def weaver_success(N, use):
    """Select with different aggregations and compare hit rates."""
    ok = 0
    for pi in range(n_te):
        base = pi * cpp
        votes = S[base:base + N]
        if use == "majority":
            vals = te_answers[te_start + base:te_start + base + N]
            counts = {}
            for v in vals:
                counts[v] = counts.get(v, 0) + 1
            idx = int(np.argmax([counts.get(v, 0) for v in vals]))
        elif use == "equal":
            idx = int(np.argmax(votes.mean(1)))
        elif use == "mom":
            sc = np.array([posterior_score(votes[j], pY_hat, tpr_hat, tnr_hat)
                           for j in range(N)])
            idx = int(np.argmax(sc))
        else:
            sc = np.array([posterior_score(votes[j], Y_te.mean(),
                                           true_tpr, true_tnr)
                           for j in range(N)])
            idx = int(np.argmax(sc))
        ok += int(te_answers[te_start + base + idx]
                  == te_golden[te_start + base + idx])
    return ok / n_te


for N in [10, 25]:
    row = " | ".join(
        "{} {:.2f}".format(s, weaver_success(N, s))
        for s in ["majority", "equal", "mom", "oracle-w"])
    print("N={}: {}".format(N, row))

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()


CN_OP = {"+": "plus", "-": "minus", "*": "times"}


def llm_sub_answer(client, a, op, b):
    """Ask the LLM to compute one sub-expression; return an integer, or None if it cannot be parsed."""
    prompt = "Compute {} {} {} and reply with the number only.".format(a, CN_OP[op], b)
    try:
        reply = client.chat([{"role": "user", "content": prompt}])
    except Exception:
        reply = "simulated output: could not connect to the API."
    m = re.search(r"-?\d+", reply)
    return int(m.group(0)) if m else None


def llm_propose_full(client, problem):
    """Ask the LLM to compute the two sub-expressions in two steps and combine them into a candidate final answer."""
    (a1, op1, b1), (a2, op2, b2) = problem["parts"]
    l = llm_sub_answer(client, a1, op1, b1)
    r = llm_sub_answer(client, a2, op2, b2)
    if l is None or r is None:
        return None, None
    ans = apply(problem["combine"], l, r)
    text = "\n".join([
        f"{a1} {op1} {b1} = {l}",
        f"{a2} {op2} {b2} = {r}",
        f"Answer = {ans}",
    ])
    return text, ans


# Pick three addition/subtraction problems so the scripted demo does not need to handle multiplication
simple_probs = []
seen = set()
for d in pool:
    parts = tuple(sorted(d["prob"]["parts"]))
    if parts in seen:
        continue
    if all(op in "+-" for (_, op, _) in d["prob"]["parts"]):
        seen.add(parts)
        simple_probs.append(d["prob"])
    if len(simple_probs) == 3:
        break

print("This run is a live API demo: the LLM gives deterministic results on addition and subtraction; under a live API the model may err, and the verifier intercepts.")
for prob in simple_probs:
    text, ans = llm_propose_full(client, prob)
    if text is None:
        print("Could not parse the LLM output; skipping.")
        continue
    _, final_ok = judge_solution(text, prob)
    score = predict_prob(orm, orm_features(text).reshape(1, -1))[0]
    print("Problem:", prob["parts"], "| gold:", prob["golden"])
    print("LLM proposed answer:", ans, "| judged:", "correct" if final_ok else "wrong",
          "| outcome verifier score: {:.3f}".format(score))

In [ ]:
# Exercise 1: complete the two reductions
step_probs = np.array([0.9, 0.8, 0.2, 0.9])


def score_product(probs):
    """Product reduction: multiply the probabilities of all steps."""
    return float(np.prod(probs))  # fill in: np.prod


def score_min(probs):
    """Min reduction: take the minimum probability."""
    return float(np.min(probs))   # fill in: np.min


assert abs(score_min(step_probs) - 0.2) < 1e-9
assert abs(score_product(step_probs) - 0.9 * 0.8 * 0.2 * 0.9) < 1e-9
print("Both reductions are correct: product reduction penalizes longer solutions; min reduction watches only the worst step.")

In [ ]:
# Exercise 2: complete HE and SE
answers = np.array([True, True, False, False, True, False, False, False])


def hard_estimation(flags):
    """Hard estimation: return 1 as soon as any continuation reaches the correct answer."""
    return int(flags.any())         # fill in: any


def soft_estimation(flags):
    """Soft estimation: return the fraction of continuations that reach the correct answer."""
    return float(flags.mean())      # fill in: mean


assert hard_estimation(answers) == 1
assert abs(soft_estimation(answers) - 3 / 8) < 1e-9
print("HE and SE are both correct: SE is frequency, HE is existence; smaller N makes SE noisier.")

In [ ]:
# Exercise 3: complete the two selectors
candidates = [
    (0.9, "A", True), (0.85, "A", True),
    (0.1, "B", False), (0.15, "B", False),
    (0.2, "B", False), (0.05, "B", False),
]


def choose_by_verifier(cands):
    """Verifier: return the final answer of the highest-scoring candidate."""
    best = max(cands, key=lambda c: c[0])  # fill in: c[0] takes the score
    return best[1]


def choose_by_majority(cands):
    """Majority vote: return the most frequent final answer."""
    from collections import Counter
    counts = Counter(c[1] for c in cands)  # fill in: c[1] takes the answer
    return max(counts, key=counts.get)


assert choose_by_verifier(candidates) == "A"
assert choose_by_majority(candidates) == "B"
print("The verifier selects A (high score), majority vote selects B (more frequent): when wrong answers cluster but have low scores, the verifier wins.")